# Module 3: 模型层 (Model Layers)

## 学习目标
- 理解 Tensor Parallelism 的基本概念
- 掌握 Linear 层的并行实现
- 理解 RMSNorm 及其融合优化
- 学习 Rotary Position Embedding (RoPE)

---

## 3.1 代码位置

模型层定义在 `python/minisgl/layers/` 目录中：

```
mini-sglang/python/minisgl/layers/
├── base.py        # 基类定义
├── linear.py      # 线性层 (支持 TP)
├── embedding.py   # 嵌入层
├── norm.py        # RMSNorm
├── rotary.py      # RoPE
└── activation.py  # 激活函数
```

## 3.2 Tensor Parallelism 基础

当模型太大无法放入单个 GPU 时，我们需要将模型分布到多个 GPU 上。

### 两种主要的并行方式:

**Column Parallel (列并行)**:
- 将权重矩阵按列切分
- 每个 GPU 持有部分列
- 输出需要 concatenate

```
        Weight [d, 4d]               GPU 0: [d, 2d]    GPU 1: [d, 2d]
     ┌─────────────────┐            ┌──────────┐      ┌──────────┐
     │                 │            │          │      │          │
     │    W_full       │  ──────►   │   W_0    │      │   W_1    │
     │                 │            │          │      │          │
     └─────────────────┘            └──────────┘      └──────────┘

X @ W_full = [X @ W_0 || X @ W_1]  (concatenate)
```

**Row Parallel (行并行)**:
- 将权重矩阵按行切分
- 每个 GPU 持有部分行
- 输出需要 all-reduce (求和)

```
        Weight [4d, d]               GPU 0: [2d, d]    GPU 1: [2d, d]
     ┌─────┐                        ┌─────┐          ┌─────┐
     │     │                        │     │          │     │
     │     │                        │ W_0 │          │ W_1 │
     │  W  │           ──────►      │     │          │     │
     │     │                        └─────┘          └─────┘
     │     │
     └─────┘

X @ W = X_0 @ W_0 + X_1 @ W_1  (all-reduce sum)
```

In [ ]:
import torch
import torch.nn.functional as F

# 模拟 Tensor Parallelism 的分布式信息
class TPInfo:
    """模拟 Tensor Parallelism 信息"""
    def __init__(self, rank: int, size: int):
        self.rank = rank  # 当前 GPU 的 rank
        self.size = size  # 总共的 GPU 数量

_TP_INFO = TPInfo(0, 1)  # 默认单 GPU

def set_tp_info(rank: int, size: int):
    global _TP_INFO
    _TP_INFO = TPInfo(rank, size)

def get_tp_info() -> TPInfo:
    return _TP_INFO

def divide_even(n: int, div: int) -> int:
    """确保 n 能被 div 整除"""
    assert n % div == 0, f"{n} is not divisible by {div}"
    return n // div

print(f"当前 TP 信息: rank={get_tp_info().rank}, size={get_tp_info().size}")

## 3.3 Linear 层实现

Mini-SGLang 中有几种不同的 Linear 层实现：

In [ ]:
class LinearTPBase:
    """Linear 层的基类，支持 Tensor Parallelism"""
    
    def __init__(
        self,
        full_isize: int,   # 完整的输入大小
        full_osize: int,   # 完整的输出大小
        local_isize: int,  # 本地 GPU 的输入大小
        local_osize: int,  # 本地 GPU 的输出大小
        has_bias: bool,
    ):
        self.full_input_size = full_isize
        self.full_output_size = full_osize
        self.local_input_size = local_isize
        self.local_output_size = local_osize
        # 本地权重大小
        self.weight = torch.empty(local_osize, local_isize)
        self.bias = torch.empty(local_osize) if has_bias else None
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.linear(x, self.weight, self.bias)

In [ ]:
class LinearColParallelMerged(LinearTPBase):
    """Column Parallel Linear，合并多个输出
    
    用于 MLP 的 gate_up_proj (合并 gate 和 up 两个投影)
    """
    
    def __init__(
        self,
        input_size: int,
        output_sizes: list,  # 多个输出大小
        has_bias: bool = False,
    ):
        tp_info = get_tp_info()
        # 每个输出都要能被 TP size 整除
        tp_output_sizes = [divide_even(size, tp_info.size) for size in output_sizes]
        output_size = sum(output_sizes)
        tp_output_size = sum(tp_output_sizes)
        super().__init__(input_size, output_size, input_size, tp_output_size, has_bias)
        
        print(f"LinearColParallelMerged:")
        print(f"  完整输出大小: {output_sizes} = {output_size}")
        print(f"  本地输出大小: {tp_output_sizes} = {tp_output_size}")

# 示例：创建 MLP 的 gate_up_proj
hidden_size = 4096
intermediate_size = 11008

# 单 GPU 情况
set_tp_info(0, 1)
print("=== 单 GPU ===")
gate_up = LinearColParallelMerged(
    input_size=hidden_size,
    output_sizes=[intermediate_size, intermediate_size],  # gate + up
    has_bias=False
)
print(f"权重形状: {gate_up.weight.shape}")

# 4 GPU 情况
print("\n=== 4 GPU TP ===")
set_tp_info(0, 4)
gate_up_tp = LinearColParallelMerged(
    input_size=hidden_size,
    output_sizes=[intermediate_size, intermediate_size],
    has_bias=False
)
print(f"权重形状: {gate_up_tp.weight.shape}")

In [ ]:
class LinearQKVMerged(LinearTPBase):
    """用于 Attention 的 QKV 投影，支持 GQA
    
    合并 Q, K, V 三个投影到一个矩阵中
    支持 Grouped Query Attention (num_qo_heads > num_kv_heads)
    """
    
    def __init__(
        self,
        hidden_size: int,
        head_dim: int,
        num_qo_heads: int,  # Query/Output heads
        num_kv_heads: int,  # Key/Value heads (GQA 时小于 num_qo_heads)
        has_bias: bool = False,
    ):
        tp_info = get_tp_info()
        
        # GQA ratio: 每个 KV head 对应几个 Q head
        GQA_ratio = divide_even(num_qo_heads, num_kv_heads)
        local_num_kv = divide_even(num_kv_heads, tp_info.size)
        
        # 完整大小: Q + K + V
        full_isize = hidden_size
        full_osize = (GQA_ratio + 2) * num_kv_heads * head_dim  # Q + K + V
        
        # 本地大小
        local_isize = hidden_size
        local_osize = (GQA_ratio + 2) * local_num_kv * head_dim
        
        super().__init__(full_isize, full_osize, local_isize, local_osize, has_bias)
        
        print(f"LinearQKVMerged (GQA ratio={GQA_ratio}):")
        print(f"  num_qo_heads={num_qo_heads}, num_kv_heads={num_kv_heads}")
        print(f"  完整输出大小: {full_osize}")
        print(f"  本地输出大小: {local_osize}")

# Llama-3 8B 的配置
print("=== Llama-3 8B Attention ===")
set_tp_info(0, 1)
qkv = LinearQKVMerged(
    hidden_size=4096,
    head_dim=128,
    num_qo_heads=32,
    num_kv_heads=8,  # GQA: 4x
    has_bias=False
)
print(f"权重形状: {qkv.weight.shape}")

# 4 GPU TP
print("\n=== 4 GPU TP ===")
set_tp_info(0, 4)
qkv_tp = LinearQKVMerged(
    hidden_size=4096,
    head_dim=128,
    num_qo_heads=32,
    num_kv_heads=8,
    has_bias=False
)
print(f"权重形状: {qkv_tp.weight.shape}")

In [ ]:
class LinearRowParallel(LinearTPBase):
    """Row Parallel Linear
    
    用于:
    - Attention 的 O projection
    - MLP 的 down projection
    
    输出需要 all-reduce 求和
    """
    
    def __init__(
        self,
        input_size: int,
        output_size: int,
        has_bias: bool = False,
    ):
        tp_info = get_tp_info()
        local_input_size = divide_even(input_size, tp_info.size)
        local_output_size = output_size
        super().__init__(input_size, output_size, local_input_size, local_output_size, has_bias)
        self._tp_size = tp_info.size
        
        print(f"LinearRowParallel:")
        print(f"  完整输入大小: {input_size}")
        print(f"  本地输入大小: {local_input_size}")
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = F.linear(x, self.weight, self.bias)
        if self._tp_size > 1:
            # 需要 all-reduce (这里只是模拟)
            print("  [执行 all-reduce]")
            # y = all_reduce(y)  # 实际需要 NCCL 通信
        return y

# 示例
print("=== O Projection (4 GPU TP) ===")
set_tp_info(0, 4)
o_proj = LinearRowParallel(
    input_size=4096,  # num_heads * head_dim
    output_size=4096,
    has_bias=False
)
print(f"权重形状: {o_proj.weight.shape}")

## 3.4 RMSNorm

RMSNorm (Root Mean Square Normalization) 是 LayerNorm 的一个简化版本，被广泛用于现代 LLM 中。

### 公式:
$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{n}\sum_{i=1}^{n} x_i^2 + \epsilon}} \cdot \gamma$$

与 LayerNorm 相比:
- 没有 mean 减法 (不需要中心化)
- 没有 bias 参数
- 计算更快

In [ ]:
class RMSNorm:
    """Root Mean Square Layer Normalization"""
    
    def __init__(self, hidden_size: int, eps: float = 1e-6):
        self.eps = eps
        self.weight = torch.ones(hidden_size)  # gamma 参数
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 计算 RMS
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        # 归一化并应用权重
        return (x / rms) * self.weight
    
    def forward_inplace(self, x: torch.Tensor) -> None:
        """原地操作版本，节省内存"""
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        x.div_(rms)
        x.mul_(self.weight)

# 测试
hidden_size = 256
norm = RMSNorm(hidden_size)

x = torch.randn(2, 10, hidden_size)  # [batch, seq_len, hidden]
y = norm.forward(x)

print(f"输入形状: {x.shape}")
print(f"输出形状: {y.shape}")
print(f"输入均值: {x.mean():.4f}, 标准差: {x.std():.4f}")
print(f"输出 RMS (应该约为 1): {torch.sqrt(y.pow(2).mean()):.4f}")

In [ ]:
class RMSNormFused:
    """融合版本的 RMSNorm，将残差连接融合进来
    
    在 Transformer 中，常见的模式是:
        hidden = RMSNorm(residual + hidden)
    
    融合后可以减少内存访问
    """
    
    def __init__(self, hidden_size: int, eps: float = 1e-6):
        self.eps = eps
        self.weight = torch.ones(hidden_size)
    
    def forward(self, x: torch.Tensor, residual: torch.Tensor = None):
        """
        返回: (normalized_output, updated_residual)
        
        如果 residual 为 None: 
            - 返回 RMSNorm(x) 和 x (x 作为新的 residual)
        
        如果 residual 不为 None:
            - 计算 x = x + residual (原地操作)
            - 返回 RMSNorm(x) 和更新后的 residual
        """
        if residual is None:
            rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
            return (x / rms) * self.weight, x
        else:
            # 融合: residual += x, 然后对 residual 做 RMSNorm
            x = x + residual  # 这里可以是原地操作
            rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
            return (x / rms) * self.weight, x

# 演示 Transformer 块中的使用模式
print("Transformer 块中的 RMSNorm 使用模式:")
print()

hidden_size = 256
norm1 = RMSNormFused(hidden_size)
norm2 = RMSNormFused(hidden_size)

# 模拟输入
x = torch.randn(1, 10, hidden_size)

# 第一次：没有 residual
hidden, residual = norm1.forward(x, residual=None)
print(f"第一次 RMSNorm 后:")
print(f"  hidden (用于 attention): {hidden.shape}")
print(f"  residual (保存用于后续): {residual.shape}")

# 模拟 attention 输出
attn_output = torch.randn_like(hidden)

# 第二次：有 residual
hidden, residual = norm2.forward(attn_output, residual=residual)
print(f"\n第二次 RMSNorm 后 (融合了 residual):")
print(f"  hidden (用于 MLP): {hidden.shape}")
print(f"  residual (更新后): {residual.shape}")

## 3.5 Rotary Position Embedding (RoPE)

RoPE 是一种相对位置编码方法，通过旋转 Query 和 Key 向量来注入位置信息。

### 核心思想:
将位置信息编码为旋转角度，使得 attention score 只依赖于相对位置。

$$q_m \cdot k_n = (R_m q) \cdot (R_n k) \propto q \cdot R_{n-m} k$$

其中 $R_m$ 是位置 $m$ 对应的旋转矩阵。

In [ ]:
import math

class RotaryEmbedding:
    """Rotary Position Embedding
    
    将位置编码为旋转角度，应用到 Q 和 K 上
    """
    
    def __init__(
        self,
        head_dim: int,
        max_position: int = 8192,
        base: float = 10000.0,
    ):
        self.head_dim = head_dim
        
        # 计算频率
        # inv_freq[i] = 1 / (base^(2i/d)) 其中 i = 0, 1, ..., d/2-1
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, dtype=torch.float) / head_dim))
        
        # 计算位置对应的角度
        positions = torch.arange(max_position, dtype=torch.float)
        # [max_position, head_dim/2]
        angles = torch.einsum("i,j->ij", positions, inv_freq)
        
        # 预计算 cos 和 sin
        self.cos_cache = angles.cos()  # [max_position, head_dim/2]
        self.sin_cache = angles.sin()  # [max_position, head_dim/2]
    
    def _rotate_half(self, x: torch.Tensor) -> torch.Tensor:
        """旋转一半维度"""
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)
    
    def forward(
        self,
        positions: torch.Tensor,  # [seq_len] 位置索引
        query: torch.Tensor,      # [seq_len, num_heads, head_dim]
        key: torch.Tensor,        # [seq_len, num_kv_heads, head_dim]
    ):
        # 获取对应位置的 cos 和 sin
        cos = self.cos_cache[positions]  # [seq_len, head_dim/2]
        sin = self.sin_cache[positions]  # [seq_len, head_dim/2]
        
        # 扩展维度以匹配 heads
        cos = torch.cat([cos, cos], dim=-1).unsqueeze(1)  # [seq_len, 1, head_dim]
        sin = torch.cat([sin, sin], dim=-1).unsqueeze(1)  # [seq_len, 1, head_dim]
        
        # 应用旋转: x * cos + rotate_half(x) * sin
        query_rotated = query * cos + self._rotate_half(query) * sin
        key_rotated = key * cos + self._rotate_half(key) * sin
        
        return query_rotated, key_rotated

# 测试
head_dim = 128
num_heads = 8
num_kv_heads = 2
seq_len = 16

rope = RotaryEmbedding(head_dim)

# 创建测试数据
positions = torch.arange(seq_len)
query = torch.randn(seq_len, num_heads, head_dim)
key = torch.randn(seq_len, num_kv_heads, head_dim)

# 应用 RoPE
query_rope, key_rope = rope.forward(positions, query, key)

print(f"Query 形状: {query.shape} → {query_rope.shape}")
print(f"Key 形状: {key.shape} → {key_rope.shape}")
print(f"\n旋转后向量范数保持不变:")
print(f"  Query: {query.norm():.4f} → {query_rope.norm():.4f}")
print(f"  Key: {key.norm():.4f} → {key_rope.norm():.4f}")

In [ ]:
# 可视化 RoPE 的效果
import matplotlib.pyplot as plt

# 创建简单的 2D 向量来可视化旋转
rope_2d = RotaryEmbedding(head_dim=2, max_position=100)

# 原始向量
original = torch.tensor([[1.0, 0.0]])  # 指向 x 轴正方向

# 不同位置的旋转
positions = [0, 10, 20, 30, 40, 50]
rotated_vectors = []

for pos in positions:
    pos_tensor = torch.tensor([pos])
    q = original.unsqueeze(1)  # [1, 1, 2]
    q_rotated, _ = rope_2d.forward(pos_tensor, q, q)
    rotated_vectors.append(q_rotated.squeeze().numpy())

# 绘图
fig, ax = plt.subplots(figsize=(8, 8))

colors = plt.cm.viridis([i / len(positions) for i in range(len(positions))])

for i, (pos, vec) in enumerate(zip(positions, rotated_vectors)):
    ax.arrow(0, 0, vec[0], vec[1], head_width=0.05, head_length=0.03, 
             fc=colors[i], ec=colors[i], label=f"pos={pos}")

ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-1.2, 1.2)
ax.set_aspect('equal')
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.legend()
ax.set_title("RoPE: 不同位置对向量的旋转效果")
plt.show()

print("观察: 随着位置增加，向量被旋转的角度越大")

## 3.6 Llama3 RoPE Scaling

Llama3 引入了一种新的 RoPE scaling 方法来支持更长的上下文长度。

In [ ]:
def llama3_rope_scaling(
    inv_freq: torch.Tensor,
    scaling_factor: float = 8.0,
    low_freq_factor: float = 1.0,
    high_freq_factor: float = 4.0,
    original_max_position: int = 8192,
) -> torch.Tensor:
    """
    Llama3 的 RoPE scaling
    
    - 高频分量 (短波长): 保持不变
    - 低频分量 (长波长): 按 scaling_factor 缩放
    - 中间频率: 平滑过渡
    """
    wave_len = 2 * math.pi / inv_freq
    
    # 简化版本: 没有平滑过渡
    if low_freq_factor == high_freq_factor:
        threshold = original_max_position / high_freq_factor
        return torch.where(
            wave_len < threshold,
            inv_freq,  # 高频保持不变
            inv_freq / scaling_factor,  # 低频缩放
        )
    
    # 完整版本: 带平滑过渡
    delta = high_freq_factor - low_freq_factor
    smooth = (original_max_position / wave_len - low_freq_factor) / delta
    smooth = torch.clamp(smooth, 0, 1)
    factor = (1 - smooth) / scaling_factor + smooth
    return factor * inv_freq

# 可视化 scaling 效果
head_dim = 128
base = 10000.0

# 原始频率
inv_freq_original = 1.0 / (base ** (torch.arange(0, head_dim, 2, dtype=torch.float) / head_dim))

# Llama3 scaling 后的频率
inv_freq_scaled = llama3_rope_scaling(inv_freq_original)

plt.figure(figsize=(10, 6))
plt.semilogy(range(len(inv_freq_original)), inv_freq_original.numpy(), 'b-', label='Original')
plt.semilogy(range(len(inv_freq_scaled)), inv_freq_scaled.numpy(), 'r--', label='Llama3 Scaled')
plt.xlabel('Dimension index')
plt.ylabel('Inverse Frequency (log scale)')
plt.title('Llama3 RoPE Scaling Effect')
plt.legend()
plt.grid(True)
plt.show()

print("观察: 低频分量 (右侧) 被缩小，使得模型可以处理更长的序列")

## 3.7 小结

### 核心要点:

1. **Tensor Parallelism**:
   - Column Parallel: 输出按列切分，结果 concatenate
   - Row Parallel: 输入按行切分，结果 all-reduce

2. **Linear 层变体**:
   - `LinearColParallelMerged`: 用于 MLP 的 gate_up
   - `LinearQKVMerged`: 合并 QKV 投影，支持 GQA
   - `LinearRowParallel`: 用于 O projection 和 down projection

3. **RMSNorm**:
   - 比 LayerNorm 更简单高效
   - 融合版本可以与残差连接合并

4. **RoPE**:
   - 相对位置编码，通过旋转实现
   - Llama3 scaling 支持更长上下文

---

**下一步**: [Module 4: Attention 机制](./04_attention_mechanism.ipynb) - 深入学习 FlashAttention 和 FlashInfer。